# Task 2.3 - YOLO Instance Segmentation (Fashionpedia)

This notebook implements **Deliverable 2 - Task 3**:

- Convert Fashionpedia COCO-style polygons to **YOLO segmentation** labels.
- Train a YOLO segmentation model on train split.
- Validate on the validation split.
- Save metrics and artifacts to compare with previous models.

Default paths match your current repository structure.

In [20]:
%load_ext autoreload
%autoreload 2
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Setup

In [21]:
if IN_COLAB:
    !pip install ultralytics pyyaml


In [22]:

from __future__ import annotations
import json
from pathlib import Path


In [29]:
# Paths and training configuration
PROJECT_ROOT =  Path('.').resolve()

TRAIN_JSON = PROJECT_ROOT / 'data/instances_attributes_train2020.json'
VAL_JSON = PROJECT_ROOT / 'data/instances_attributes_val2020.json'
TRAIN_IMAGES = PROJECT_ROOT / 'data/train'
VAL_IMAGES = PROJECT_ROOT / 'data/validation'
# Same dense masks as Task 2 semantic runs (for comparable mDice / mIoU)
VAL_MASK_DIR = PROJECT_ROOT / 'data/segmentations_val'
TASK2_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'task2'

DATA_DIR = PROJECT_ROOT / 'data'
YOLO_DATASET_DIR = PROJECT_ROOT / 'yolo_dataset/'

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'task3'

MODEL_NAME = 'yolov8n-seg.pt'
SEED = 42
IMG_SIZE = 384

run_name = 'yolo_seg_res'


In [24]:


if not YOLO_DATASET_DIR.exists():
    if IN_COLAB:
      from google.colab import drive
      drive.mount('/content/drive')
      !cp /content/drive/MyDrive/OR/Deliverable\ 2/yolo_dataset.zip /content/
      !unzip -q /content/yolo_dataset.zip -d /content/

      if not YOLO_DATASET_DIR.exists():
          print("Dataset not found. Make sure the zip exists in the mounted drive")
    else:
        print("Dataset not found. Converting coco style dataset to yolo format")
        from yolo_instance_seg_helpers import create_yolo_formatted_dataset

        create_yolo_formatted_dataset(DATA_DIR=DATA_DIR, YOLO_DATASET_DIR=YOLO_DATASET_DIR, TRAIN_JSON=TRAIN_JSON, VAL_JSON=VAL_JSON, TRAIN_IMAGES=TRAIN_IMAGES, VAL_IMAGES=VAL_IMAGES)
        print("Dataset created. Setup complete")
else:
    print("Dataset setup complete")

Dataset setup complete


# Training

In [ ]:
# 3) Train YOLO instance segmentation
from ultralytics import YOLO

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(YOLO_DATASET_DIR / "fashionpedia_yolo_seg.yaml"),
    seed=SEED,
    imgsz=IMG_SIZE,
    project=str(OUTPUT_DIR / 'runs'),
    name=run_name,
    val=True,
)

print('Training finished.')
print(f'Train run dir: {train_results.save_dir}')

In [46]:
from experiment_utils import ensure_dir
from ultralytics import YOLO
from yolo_semantic_eval import evaluate_yolo, save_yolo_outputs

DATASET_YAML = YOLO_DATASET_DIR / 'fashionpedia_yolo_seg.yaml'

best_checkpoint = OUTPUT_DIR / "runs/yolo_seg_res/weights/best.pt"
if not best_checkpoint.exists():
    raise FileNotFoundError(f'Best checkpoint not found: {best_checkpoint}')

best_model = YOLO(str(best_checkpoint))
val_results = best_model.val(
    data=str(DATASET_YAML),
    project=str(OUTPUT_DIR / 'runs' / run_name),
)

metrics_payload = {
    'box_map50': float(getattr(val_results.box, 'map50', 0.0)),
    'box_map50_95': float(getattr(val_results.box, 'map', 0.0)),
    'seg_map50': float(getattr(val_results.seg, 'map50', 0.0)),
    'seg_map50_95': float(getattr(val_results.seg, 'map', 0.0)),
}

metrics_output = Path(val_results.save_dir) / 'metrics_summary.json'
ensure_dir(metrics_output.parent)
with metrics_output.open('w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2)

print(f'Saved to: {metrics_output}')

# Dense pixel metrics comparable to train_eval.validate_one_epoch (Task 2 CSV columns)
sem_metrics = evaluate_yolo(
    weights=best_checkpoint,
    val_json=VAL_JSON,
    val_image_dir=VAL_IMAGES,
    val_mask_dir=VAL_MASK_DIR,
    imgsz=IMG_SIZE,
    annotation_json_for_class_names=TRAIN_JSON,
    conf=0.25,
)


run_tag = f"{Path(MODEL_NAME).stem}_res_yolo_seg"
extra = {
    'model_name': Path(MODEL_NAME).stem,
    'seg_map50': metrics_payload['seg_map50'],
    'seg_map50_95': metrics_payload['seg_map50_95'],
    'box_map50': metrics_payload['box_map50'],
    'box_map50_95': metrics_payload['box_map50_95'],
    'history_csv': str(OUTPUT_DIR / "runs" / run_name / "results.csv"),
}


paths = save_yolo_outputs(
    sem_metrics,
    run_name=run_tag,
    weights_path=best_checkpoint,
    output_dir=OUTPUT_DIR / "runs" / run_name,
    extra_summary_fields=extra,
)


print('Task 2–aligned val (dense):', {k: sem_metrics[k] for k in ('val_pixel_acc_ex_bg', 'val_mdice_ex_bg', 'val_miou_ex_bg', 'num_val_images')})
print('Per-class CSV:', paths['per_class_csv'])
print('Aligned metrics JSON:', paths['metrics_json'])


Ultralytics 8.4.26 🚀 Python-3.13.2 torch-2.11.0 CPU (Apple M2 Pro)
YOLOv8n-seg summary (fused): 86 layers, 3,267,034 parameters, 0 gradients, 11.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 236.0±100.1 MB/s, size: 78.1 KB)
val: Scanning /Users/pandelissymeonidis/Projects/MAI/object-recognition/Fashion_Segmentation/yolo_dataset/labels/val.cache... 1158 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1158/1158 285.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 73/73 1.0it/s 1:120.8sss
                   all       1158       8495      0.594      0.364      0.384      0.279       0.58      0.326      0.333      0.186
         shirt, blouse        102        102       0.46      0.441      0.472      0.382       0.44      0.373      0.373      0.225
top, t-shirt, sweatshirt        463        477      0.661      0.727      0.757      0.582      0.657      0.69